# Pipeline

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [44]:
import torch
import pandas as pd
import os
import sys

from pathlib import Path

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    TASK_STATEMENTS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_OUTPUT_PATH,
    TIMEZONES_OUTPUT_PATH,
    TASK_MAPPING_OUTPUT_PATH,
    LABOR_TRANSFER_OUTPUT_FILE,
    JOB_ZONES_PATH,
    ExecutionMode,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
)
from validation import (
    work_related_metrics,
    agreement_rate,
    print_metrics,
)

In [3]:
# Device setup
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [4]:
# API client setup
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

In [5]:
execution_mode = ExecutionMode.DIRECT

## 2. Data Loading

In [6]:
# TODO: When we decide on the validation sample, swap out the
# sample_conversations_df with that

In [7]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [8]:
sample_df = sample_conversations(english_conversations)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (58777, 14)


In [9]:
sample_conversations_df = preprocess_conversations(sample_df)
print(f"After dedup: {sample_conversations_df.shape}")
sample_conversations_df.head(2)

After dedup: (52489, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


In [10]:
sample_conversations_df = sample_conversations_df[:50]
sample_conversations_df.shape

(50, 5)

## 3. Work-Related Conversation Filtering

In [22]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)

sample_conversations_df["is_work_related_model"] = answers_df[
    "is_work_related_model"
].values

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

Running 50 direct Work Related calls...


Work Related:  18%|█▊        | 9/50 [00:59<03:39,  5.35s/it]

Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 seconds...
Rate limit hit (attempt 2). Retrying in 2.00 seconds...
Rate limit hit (attempt 1). Retrying in 1.00 seconds...
Rate limit hit (attempt 3). Retrying in 4.00 sec

Work Related: 100%|██████████| 50/50 [05:07<00:00,  6.15s/it]

Work-related conversations: (17, 6)


,conversation,timestamp,country,state,hashed_ip,is_work_related_model
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...,Yes
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes


In [23]:
sample_conversations_df["is_work_related_model"].value_counts()

is_work_related_model
No       23
Yes      17
Maybe    10
Name: count, dtype: int64

In [ ]:
metrics = work_related_metrics(
    y_true=sample_conversations_df["is_work_related_human"],
    y_pred=sample_conversations_df["is_work_related_model"],
)
print_metrics(metrics)

## 4. Timezone conversion

In [24]:
if not TIMEZONES_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_OUTPUT_PATH)

work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

Error geocoding location for state 'Royal Kensington and Chelsea' and country 'United Kingdom': HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Royal+Kensington+and+Chelsea%2C+United+Kingdom&format=json&limit=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='nominatim.openstreetmap.org', port=443) at 0xe21b8ea20>, 'Connection to nominatim.openstreetmap.org timed out. (connect timeout=1)'))
Geocoding query: 'Khyber Pakhtunkhwa, Pakistan' -> Location: خیبر پختونخوا, پاکستان
Found location for query 'Khyber Pakhtunkhwa, Pakistan': خیبر پختونخوا, پاکستان (lat: 33.712802, lng: 71.2678805)
Geocoding query: 'nan' -> Location: ننگرهار ولايت, افغانستان
Found location for query 'nan': ننگرهار ولايت, افغانستان (lat: 34.220389, lng: 70.3800314)
Geocoding query: 'Capital Region, Denmark' -> Location: Sendiráð Danmerkur, 29, Hverfisgata, Austurbær, Miðborg, Reykjavíkurborg, Höfuðborgarsvæðið, 101, Ísland
Found location for query 'Capi

In [25]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

,timestamp,timezone,timestamp_local
1,2024-10-01 15:01:08+00:00,Asia/Karachi,2024-10-01 20:01:08+05:00
2,2024-11-04 01:14:03+00:00,Asia/Kabul,2024-11-04 05:44:03+04:30
3,2024-11-05 23:54:06+00:00,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00


## 5. Task Mapping

In [26]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Management Occupations
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Management Occupations


In [29]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

Running 15 direct Profession mapping calls via OpenRouter...


Profession Mapping: 100%|██████████| 15/15 [03:22<00:00, 13.50s/it]


Running 15 direct Task mapping calls via OpenRouter...


Task Mapping: 100%|██████████| 15/15 [04:29<00:00, 17.96s/it]

Task mapped conversations: (15, 3)


,conversation,professions,tasks
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Exe...",[Editors:Read copy or proof to detect and corr...
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",[Interpreters and Translators:Translate messag...


In [31]:
task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[0] if pd.notnull(x) else None
)
task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[1] if pd.notnull(x) else None
)

print(f"After consensus filter: {task_mapped_df.shape}")
task_mapped_df.head(2)

Error processing items: ['Identify System 1 and System 2 Thinking Requirements', 'Apply Step-by-Step Problem Solving with Creativity and Metacognitive Reflection for System 2 Sections', 'Generate Hypotheses with Confidence and Creative Scores', 'Anticipate Future Steps and Obstacles', 'Reflect and Capture Insights']
Error processing items: ['Correct errors by making appropriate changes and rechecking the program to ensure that the desired results are produced.', 'Write, update, and maintain computer programs or software packages to handle specific jobs such as tracking inventory, storing or retrieving data, or controlling other equipment.', 'Evaluate code to ensure that it is valid, is properly structured, meets industry standards, and is compatible with browsers, devices, or operating systems.', 'Write supporting code for Web applications or Web sites.', 'Identify problems uncovered by testing or customer feedback, and correct problems or refer problems to appropriate personnel for co

,conversation,professions,tasks,job_title,selected_task
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Exe...",Editors:Read copy or proof to detect and corre...,Editors,Read copy or proof to detect and correct error...
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",Interpreters and Translators:Translate message...,Interpreters and Translators,Translate messages simultaneously or consecuti...


In [33]:
task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

task_mapped_df = task_mapped_df.merge(
    work_related_df.drop(columns=["conversation"]), on="conversation_str", how="inner"
)

task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final task mapped DataFrame: (9, 12)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Exe...",Editors:Read copy or proof to detect and corre...,Editors,Read copy or proof to detect and correct error...,2024-10-01 15:01:08+00:00,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes,Asia/Karachi,2024-10-01 20:01:08+05:00
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",Interpreters and Translators:Translate message...,Interpreters and Translators,Translate messages simultaneously or consecuti...,2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30


In [ ]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="selected_task_human_eval",
)
print(agreement)

## 6. Labor Transfer Analysis

In [49]:
if not LABOR_TRANSFER_OUTPUT_FILE.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_OUTPUT_FILE, index=False
    )
else:
    labor_transfer_labels = pd.read_csv(LABOR_TRANSFER_OUTPUT_FILE)["label"].tolist()

print(f"Labor transfer labels: {len(labor_transfer_labels)}")
task_mapped_df["labor_transfer"] = labor_transfer_labels
print(f"Labor transfer labels assigned: {task_mapped_df.shape}")

Labor transfer labels: 9
Labor transfer labels assigned: (9, 13)


In [50]:
task_mapped_df = expand_labor_transfer_labels(
    df=task_mapped_df, label_column="labor_transfer"
)
print(f"Final DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final DataFrame: (9, 20)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Exe...",Editors:Read copy or proof to detect and corre...,Editors,Read copy or proof to detect and correct error...,2024-10-01 15:01:08+00:00,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes,Asia/Karachi,2024-10-01 20:01:08+05:00,consumer,good,LT1,low_stakes,editor,NaN,User asks ChatGPT to correct a sentence for sp...,high
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",Interpreters and Translators:Translate message...,Interpreters and Translators,Translate messages simultaneously or consecuti...,2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30,consumer,good,LT1,low_stakes,translator,NaN,User requested translation of a specific Engli...,high


In [ ]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="labor_transfer_human_eval",
)
print(agreement)

In [ ]:
# final_df.to_csv()

In [ ]:
# TODO:
# - [ ] Aggiungere analisi nel tempo (mesi/ore)
# - [ ] Aggiungere analisi per hashed_ip
# - [ ] Codice dei plots
# - [ ] Usare dei path diversi per i risultati di validazione (es. una cartella validation_results/)

# 7. Job Zones

In [48]:
job_zones_pdf = pd.read_excel(JOB_ZONES_PATH)
job_zones_pdf.head(2)

,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst


In [52]:
task_mapped_df = task_mapped_df.merge(
    job_zones_pdf[["Title", "Job Zone"]],
    left_on="job_title",
    right_on="Title",
    how="left",
)
task_mapped_df = task_mapped_df.drop(columns=["Title"])
print(f"After merging job zones: {task_mapped_df.shape}")
task_mapped_df.head(2)

After merging job zones: (9, 21)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,...,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence,Job Zone
0,"[{'role': 'user', 'content': 'sir followings a...","[Editors, Technical Writers, Copy Writers, Exe...",Editors:Read copy or proof to detect and corre...,Editors,Read copy or proof to detect and correct error...,2024-10-01 15:01:08+00:00,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes,...,2024-10-01 20:01:08+05:00,consumer,good,LT1,low_stakes,editor,NaN,User asks ChatGPT to correct a sentence for sp...,high,4.0
1,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",Interpreters and Translators:Translate message...,Interpreters and Translators,Translate messages simultaneously or consecuti...,2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,...,2024-11-04 05:44:03+04:30,consumer,good,LT1,low_stakes,translator,NaN,User requested translation of a specific Engli...,high,4.0
